# **Penting**

*   Pada Notebook ini, Anda hanya perlu mengerjakan code pada bagian **logic.py** saja. Anda tidak diwajibkan untuk mengubah atau menambahkan **app.py** yang digunakan untuk membangun interface Streamlit.
*   Namun, jika Anda memiliki preferensi lain atau ingin mengubah struktur code pada logic ataupun pada interface Streamlit, itu **DIPERSILAHKAN** saja, tetapi pastikan untuk memenuhi kriteria yang telah ditetapkan pada intruksi submission
*   Jika Anda tidak ingin mengubah apapun dan ingin mengikuti template, tugas Anda hanyalah melengkapi code yang rumpang pada bagian yang sudah ditandai "________" saja.



# **Prepare Dependencies**

In [1]:
!pip install -q streamlit==1.32.2 streamlit-drawable-canvas==0.9.3 pyngrok diffusers transformers accelerate safetensors pillow scipy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is inc

In [2]:
from pyngrok import ngrok
import subprocess
import time

# **Streamlit**

## logic.py (**Basic**)

In [3]:
%%writefile logic.py
import gc
import torch
import numpy as np
import streamlit as st
from PIL import Image, ImageFilter
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionInpaintPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
)

BASE_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
INPAINT_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


@st.cache_resource
def load_text2img_pipeline():
    device = get_device()
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=dtype,
        safety_checker=None
    ).to(device)
    return pipe


def generate_basic_image(prompt, negative_prompt="", guidance_scale=7.5, num_inference_steps=30, seed=222):
    device = get_device()
    pipe = load_text2img_pipeline()
    generator = torch.Generator(device=device).manual_seed(int(seed))
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(num_inference_steps),
        generator=generator,
        width=512,
        height=512
    )
    return result.images[0]

Writing logic.py


## logic.py (**Skilled**)

In [4]:
%%writefile -a logic.py


def load_scheduler(pipe, scheduler_name):
    if scheduler_name == "Euler A":
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "DPM++":
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "DDIM":
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    return pipe


def generate_images(
    prompt,
    negative_prompt="",
    guidance_scale=7.5,
    num_inference_steps=30,
    num_images=1,
    scheduler_name="DPM++",
    seed=222
):
    device = get_device()
    pipe = load_text2img_pipeline()
    pipe = load_scheduler(pipe, scheduler_name)

    generators = [
        torch.Generator(device=device).manual_seed(int(seed) + i)
        for i in range(int(num_images))
    ]

    result = pipe(
        prompt=[prompt] * int(num_images),
        negative_prompt=[negative_prompt] * int(num_images),
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(num_inference_steps),
        generator=generators,
        width=512,
        height=512
    )
    return result.images


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Appending to logic.py


## logic.py (**Advanced**)

In [5]:
%%writefile -a logic.py


@st.cache_resource
def load_inpaint_pipeline():
    device = get_device()
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        INPAINT_MODEL_ID,
        torch_dtype=dtype,
        safety_checker=None
    ).to(device)
    return pipe


def inpaint_image(
    image,
    mask,
    prompt,
    negative_prompt="",
    strength=0.95,
    guidance_scale=8.5,
    num_inference_steps=30,
    seed=9
):
    device = get_device()
    pipe_inpaint = load_inpaint_pipeline()

    if image.mode != "RGB":
        image = image.convert("RGB")
    if mask.mode != "L":
        mask = mask.convert("L")
    if image.size != mask.size:
        mask = mask.resize(image.size, resample=Image.NEAREST)

    generator = torch.Generator(device=device).manual_seed(int(seed))
    result = pipe_inpaint(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        strength=float(strength),
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(num_inference_steps),
        generator=generator,
        width=image.width,
        height=image.height
    )
    return result.images[0]


def prepare_outpainting_canvas(image, mode="Zoom Out (All Directions)", direction="Right", expand_pixels=128):
    if image.mode != "RGB":
        image = image.convert("RGB")

    w, h = image.size
    if mode == "Zoom Out (All Directions)":
        new_w = w + (expand_pixels * 2)
        new_h = h + (expand_pixels * 2)
        paste_x = expand_pixels
        paste_y = expand_pixels
    else:
        if direction in ["Left", "Right"]:
            new_w, new_h = w + expand_pixels, h
        else:
            new_w, new_h = w, h + expand_pixels

        paste_x = expand_pixels if direction == "Left" else 0
        paste_y = expand_pixels if direction == "Top" else 0

    new_w -= new_w % 8
    new_h -= new_h % 8

    bg = image.resize((new_w, new_h), resample=Image.BICUBIC)
    bg = bg.filter(ImageFilter.GaussianBlur(radius=50))
    canvas = bg.copy()
    canvas.paste(image, (paste_x, paste_y))

    mask = Image.new("L", (new_w, new_h), 255)
    inner_box = Image.new("L", (w, h), 0)
    mask.paste(inner_box, (paste_x, paste_y))

    return canvas, mask


def outpaint_image(
    image,
    prompt,
    negative_prompt="",
    mode="Zoom Out (All Directions)",
    direction="Right",
    expand_pixels=128,
    strength=0.95,
    guidance_scale=8.5,
    num_inference_steps=30,
    seed=222
):
    canvas, mask = prepare_outpainting_canvas(
        image=image,
        mode=mode,
        direction=direction,
        expand_pixels=int(expand_pixels)
    )
    output_img = inpaint_image(
        image=canvas,
        mask=mask,
        prompt=prompt,
        negative_prompt=negative_prompt,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        seed=seed
    )
    return output_img, mask

Appending to logic.py


## app.py
Anda **TIDAK perlu mengubah atau menambahkan** apapun pada file **app.py** ini, cukup **jalankan** saja.

In [6]:
%%writefile app.py
import io
import numpy as np
import streamlit as st
from PIL import Image
from streamlit_drawable_canvas import st_canvas
from logic import (
    generate_images,
    inpaint_image,
    outpaint_image,
    clear_memory,
)

st.set_page_config(
    page_title="Generative Image Studio - BFGAI",
    page_icon="🎨",
    layout="wide"
)

if "generated_images" not in st.session_state:
    st.session_state["generated_images"] = []
if "active_image" not in st.session_state:
    st.session_state["active_image"] = None

with st.sidebar:
    st.header("⚙️ System & Memory")
    st.caption("Model: Stable Diffusion v1.5")
    if st.button("🧹 Clear Memory", use_container_width=True):
        clear_memory()
        st.success("Memori RAM & VRAM GPU berhasil dibersihkan!")

    st.divider()
    if st.session_state["active_image"] is not None:
        st.subheader("🖼️ Active Image for Edit")
        st.image(st.session_state["active_image"], use_column_width=True)
    else:
        st.info("Generate gambar terlebih dahulu pada Tab Text-to-Image untuk langsung diedit di Tab Inpainting & Outpainting.")

st.title("🎨 Generative Image Suite (Stable Diffusion 1.5)")

tab_gen, tab_inpaint, tab_outpaint = st.tabs([
    "✨ Text-to-Image Generation",
    "🖌️ Inpainting (Canvas Mask)",
    "🔭 Outpainting & Zoom Out"
])

# ==========================================
# TAB 1: TEXT-TO-IMAGE GENERATION
# ==========================================
with tab_gen:
    col_ctrl, col_out = st.columns([1, 1.3], gap="large")

    with col_ctrl:
        st.subheader("Parameter Generasi")
        prompt = st.text_area(
            "Prompt",
            value="2d digital art illustration of an astronaut in a white spacesuit standing on the moon surface, planet earth in starry sky, clean vector art",
            height=110
        )
        negative_prompt = st.text_area(
            "Negative Prompt",
            value="photorealistic, realistic, photograph, 3d render, messy, blurry, low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration",
            height=90
        )

        c1, c2 = st.columns(2)
        with c1:
            scheduler_name = st.selectbox(
                "Scheduler",
                options=["Euler A", "DPM++", "DDIM"],
                index=1
            )
            guidance_scale = st.slider(
                "Guidance Scale (CFG)",
                min_value=1.0,
                max_value=20.0,
                value=8.5,
                step=0.5
            )
        with c2:
            num_images = st.selectbox(
                "Jumlah Gambar (Grid 2x2)",
                options=[1, 2, 4],
                index=2
            )
            num_inference_steps = st.slider(
                "Inference Steps",
                min_value=10,
                max_value=50,
                value=30,
                step=5
            )

        seed = st.number_input("Seed", min_value=0, max_value=999999, value=222, step=1)
        btn_generate = st.button("🚀 Generate Image", type="primary", use_container_width=True)

    with col_out:
        st.subheader("Hasil Generasi")
        if btn_generate:
            if not prompt.strip():
                st.warning("Masukkan prompt terlebih dahulu.")
            else:
                with st.spinner("Sedang membuat gambar dengan Stable Diffusion v1.5..."):
                    imgs = generate_images(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        guidance_scale=guidance_scale,
                        num_inference_steps=num_inference_steps,
                        num_images=num_images,
                        scheduler_name=scheduler_name,
                        seed=seed
                    )
                    st.session_state["generated_images"] = imgs
                    st.session_state["active_image"] = imgs[0]

        if st.session_state["generated_images"]:
            imgs = st.session_state["generated_images"]
            if len(imgs) == 1:
                st.image(imgs[0], caption=f"Generated (Seed={seed})", use_column_width=True)
            else:
                for row_start in range(0, len(imgs), 2):
                    grid_cols = st.columns(2)
                    for idx_offset in range(2):
                        img_idx = row_start + idx_offset
                        if img_idx < len(imgs):
                            with grid_cols[idx_offset]:
                                st.image(
                                    imgs[img_idx],
                                    caption=f"Image #{img_idx + 1} (Seed={int(seed) + img_idx})",
                                    use_column_width=True
                                )
                                if st.button(f"Pilih Gambar #{img_idx + 1} untuk Edit", key=f"select_img_{img_idx}"):
                                    st.session_state["active_image"] = imgs[img_idx]
                                    st.rerun()

# ==========================================
# TAB 2: INPAINTING DENGAN DRAWABLE CANVAS
# ==========================================
with tab_inpaint:
    st.subheader("🖌️ Inpainting dengan Masking Interaktif")
    if st.session_state["active_image"] is None:
        st.warning("Belum ada gambar aktif. Silakan generate gambar di Tab Text-to-Image terlebih dahulu.")
    else:
        base_img = st.session_state["active_image"].resize((512, 512))
        col_canvas, col_inp_ctrl = st.columns([1.1, 1], gap="large")

        with col_canvas:
            st.markdown("**Gambar Mask pada Area yang Ingin Diubah:**")
            stroke_width = st.slider("Ukuran Kuas (Brush Size)", min_value=5, max_value=60, value=25)
            canvas_result = st_canvas(
                fill_color="rgba(255, 255, 255, 0.5)",
                stroke_width=stroke_width,
                stroke_color="#FFFFFF",
                background_image=base_img,
                update_streamlit=True,
                height=512,
                width=512,
                drawing_mode="freedraw",
                key="inpaint_canvas"
            )

        with col_inp_ctrl:
            inpaint_prompt = st.text_area(
                "Prompt Inpainting",
                value="2d digital art illustration of a broken crashed satellite with solar panels on the moon, clean vector art",
                height=100
            )
            inpaint_neg = st.text_area(
                "Negative Prompt Inpainting",
                value="photorealistic, blurry, low quality, messy, ugly, watermark",
                height=80
            )
            inpaint_strength = st.slider("Strength", min_value=0.3, max_value=1.0, value=0.95, step=0.05)
            inpaint_cfg = st.slider("Guidance Scale", min_value=1.0, max_value=20.0, value=9.0, step=0.5, key="inp_cfg")
            inpaint_steps = st.slider("Inference Steps", min_value=15, max_value=50, value=30, step=5, key="inp_steps")
            inpaint_seed = st.number_input("Seed Inpainting", min_value=0, max_value=999999, value=9, step=1)

            btn_inpaint = st.button("✨ Jalankan Inpainting", type="primary", use_container_width=True)

            if btn_inpaint:
                if canvas_result.image_data is None or np.max(canvas_result.image_data[:, :, 3]) == 0:
                    st.warning("Silakan coret area mask pada gambar di sebelah kiri terlebih dahulu.")
                else:
                    alpha_channel = canvas_result.image_data[:, :, 3]
                    mask_array = np.where(alpha_channel > 0, 255, 0).astype(np.uint8)
                    mask_pil = Image.fromarray(mask_array, mode="L")

                    with st.spinner("Memproses Inpainting..."):
                        inpainted_res = inpaint_image(
                            image=base_img,
                            mask=mask_pil,
                            prompt=inpaint_prompt,
                            negative_prompt=inpaint_neg,
                            strength=inpaint_strength,
                            guidance_scale=inpaint_cfg,
                            num_inference_steps=inpaint_steps,
                            seed=inpaint_seed
                        )
                        st.session_state["inpainted_result"] = inpainted_res
                        st.session_state["inpainted_mask"] = mask_pil

            if "inpainted_result" in st.session_state:
                st.divider()
                r1, r2 = st.columns(2)
                with r1:
                    st.image(st.session_state["inpainted_mask"], caption="Mask", use_column_width=True)
                with r2:
                    st.image(st.session_state["inpainted_result"], caption="Hasil Inpainting", use_column_width=True)
                if st.button("📌 Gunakan Hasil Ini untuk Outpainting", use_container_width=True):
                    st.session_state["active_image"] = st.session_state["inpainted_result"]
                    st.rerun()

# ==========================================
# TAB 3: OUTPAINTING & ZOOM OUT
# ==========================================
with tab_outpaint:
    st.subheader("🔭 Outpainting & Zoom Out")
    if st.session_state["active_image"] is None:
        st.warning("Belum ada gambar aktif. Silakan generate gambar di Tab Text-to-Image terlebih dahulu.")
    else:
        col_out_ctrl, col_out_res = st.columns([1, 1.3], gap="large")

        with col_out_ctrl:
            outpaint_mode = st.selectbox(
                "Mode Perluasan Kanvas",
                options=["Zoom Out (All Directions)", "One-Sided Outpainting"]
            )
            direction = "Right"
            if outpaint_mode == "One-Sided Outpainting":
                direction = st.selectbox("Arah Perluasan", options=["Right", "Left", "Top", "Bottom"])

            expand_px = st.slider("Ukuran Perluasan (Pixel)", min_value=64, max_value=256, value=128, step=64)
            outpaint_prompt = st.text_area(
                "Prompt Outpainting / Zoom Out",
                value="wide angle 2d digital art illustration of an astronaut on the moon surface, vast lunar craters, planet earth and stars in dark sky, clean vector art",
                height=100
            )
            outpaint_neg = st.text_area(
                "Negative Prompt",
                value="photorealistic, blurry, low quality, border, frame, cut off, messy",
                height=80,
                key="out_neg"
            )
            out_cfg = st.slider("Guidance Scale", min_value=1.0, max_value=20.0, value=8.5, step=0.5, key="out_cfg")
            out_steps = st.slider("Inference Steps", min_value=15, max_value=50, value=30, step=5, key="out_steps")
            out_seed = st.number_input("Seed Outpainting", min_value=0, max_value=999999, value=222, step=1)

            btn_outpaint = st.button("🌌 Jalankan Outpainting / Zoom Out", type="primary", use_container_width=True)

        with col_out_res:
            if btn_outpaint:
                with st.spinner("Memperluas kanvas dan merender Outpainting..."):
                    out_res, out_mask = outpaint_image(
                        image=st.session_state["active_image"],
                        prompt=outpaint_prompt,
                        negative_prompt=outpaint_neg,
                        mode=outpaint_mode,
                        direction=direction,
                        expand_pixels=expand_px,
                        strength=0.95,
                        guidance_scale=out_cfg,
                        num_inference_steps=out_steps,
                        seed=out_seed
                    )
                    st.session_state["outpainted_result"] = out_res
                    st.session_state["outpainted_mask"] = out_mask

            if "outpainted_result" in st.session_state:
                m_col, res_col = st.columns([1, 2])
                with m_col:
                    st.image(st.session_state["outpainted_mask"], caption="Area Mask", use_column_width=True)
                with res_col:
                    st.image(
                        st.session_state["outpainted_result"],
                        caption=f"Hasil {outpaint_mode} ({st.session_state['outpainted_result'].size})",
                        use_column_width=True
                    )
                buf = io.BytesIO()
                st.session_state["outpainted_result"].save(buf, format="PNG")
                st.download_button(
                    label="⬇️ Download Hasil Gambar (PNG)",
                    data=buf.getvalue(),
                    file_name="hasil_generative_studio.png",
                    mime="image/png",
                    use_container_width=True
                )

Writing app.py


# **Menggunakan *Ngrok* Untuk Deployment**

## **Konfigurasi Autentikasi Ngrok dan Menjalankan Aplikasi Streamlit**
Cell ini digunakan untuk mengatur *authentication token Ngrok* dan menjalankan aplikasi Streamlit secara lokal. Token diperlukan agar *Ngrok* dapat membuat tunnel dengan akun pengguna. Setelah token diatur, aplikasi Streamlit dijalankan menggunakan subprocess sehingga server lokal aktif di background.

Apabila Anda belum mengetahui cara mendapatkan *authentication token Ngrok* milik Anda sendiri, silahkan baca pada bagian **tips and tricks submission**.

In [7]:
NGROK_AUTH_TOKEN = "3J8tqvgXyTiFdiWzwzOrSe2tDw3_84j2dZKs5PCtqmwfNWXvX"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], check=False)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

## **Membuat Public URL**
Cell ini membuat *tunnel Ngrok* ke port lokal tempat Streamlit berjalan (default: 8501). *Ngrok* kemudian menghasilkan public URL yang bisa diakses dari internet, sehingga aplikasi Streamlit dapat dibuka tanpa harus berada di jaringan lokal yang sama.

In [8]:
public_url = ngrok.connect(8501)
print(f"Aplikasi Streamlit aktif!\n{public_url.public_url}")

Aplikasi Streamlit aktif!
https://stubble-outspoken-squatted.ngrok-free.dev


Apabila Anda mengalami limit endpoint usage pada Ngrok, jalankan hidden cell di bawah ini!

## **Menutup Semua Tunnel Ngrok yang Aktif**
Cell ini digunakan untuk menghentikan seluruh koneksi Ngrok yang masih aktif.

In [ ]:
ngrok.kill()
print("All active ngrok tunnels have been closed.")